# Taller — Construye y ajusta tu propio clasificador de imágenes

Este taller comienza **después de la teoría de CNN y del notebook guiado de transfer learning**.

La meta no es memorizar la API de Keras. La meta es tomar decisiones sobre un problema real: construir datos propios, adaptar un modelo preentrenado y analizar qué tan bien funciona.

---

## Organización de la actividad

| Momento | Tiempo estimado | Meta |
|---|---:|---|
| **Teoría + notebook guiado** | Se realiza antes de este taller | Comprender CNN, transfer learning y fine-tuning |
| **Trabajo en clase** | ~60 min | Construir el dataset y entrenar el primer modelo con la base congelada |
| **Trabajo en casa** | ~1–2 h | Evaluar, hacer fine-tuning, comparar y analizar errores |

### Regla importante

Durante el trabajo en clase usaremos **train** y **validation**.  
El conjunto de **test** se reservará para el trabajo en casa, cuando hagamos la evaluación final.

---

## Entrega final

Debes entregar **este mismo notebook ejecutado**, incluyendo:

- definición del problema y estrategia de búsqueda;
- dataset propio;
- primer entrenamiento con transfer learning;
- evaluación en test;
- fine-tuning;
- comparación antes/después;
- matriz de confusión;
- análisis de al menos 5 errores;
- conclusiones.

> **Versión completada sobre la guía original.** Se conservan las 52 celdas originales, sus identificadores, consignas y orden. Las soluciones aparecen en las celdas de código y en las respuestas añadidas debajo de las preguntas.
>
> Fuente: [EAFIT SI3003 — notebook original](https://github.com/EAFIT-IA/si3003-artificial-intelligence/blob/dfbcd874eb8bc27a3f3c258b0e7fdc41a6d9dbd3/notebooks/lecture8/transfer_learning/taller_transfer_learning_datos_propios_keras3.ipynb), commit `dfbcd874eb8bc27a3f3c258b0e7fdc41a6d9dbd3`.
>
> **Estado real:** código completado y funciones auxiliares probadas sin conexión. No se han recolectado imágenes ni entrenado modelos en esta entrega. Los resultados, cinco errores y observaciones de limpieza requieren ejecutar con datos reales; no se sustituyen por cifras o experiencias inventadas.
>
> **Cómo ejecutar:** Colab con GPU, activar `INSTALAR_DEPENDENCIAS` si faltan paquetes, reiniciar el kernel, configurar/recolectar las tres clases, inspeccionar y registrar exclusiones, marcar `DATASET_REVISADO=True` y continuar. Guarda el notebook con sus salidas antes de entregarlo. Se conservan todos los enunciados “TODO” del profesor; no hay valores `TODO` sin resolver en el código.


## 0. Preparación del entorno

La búsqueda de imágenes se realizará con `ddgs`. Si estás en Google Colab, ejecuta la siguiente celda.

In [ ]:
# Ejecutar solo si faltan dependencias; después reinicia el kernel.
INSTALAR_DEPENDENCIAS = False
if INSTALAR_DEPENDENCIAS:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install",
        "tensorflow>=2.16,<3", "keras>=3,<4", "ddgs>=9,<10",
        "requests>=2.32,<3", "Pillow>=10,<13", "numpy>=1.26,<3",
        "matplotlib>=3.8,<4", "scikit-learn>=1.5,<2", "pandas>=2,<3"])
else:
    print("Instalación desactivada; se utilizará el entorno existente.")

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, io, json, random, shutil, time, uuid, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from PIL import Image, ImageOps
from IPython.display import display, Markdown

try:
    import tensorflow as tf
    import keras
    from keras import layers
except ImportError as exc:
    raise RuntimeError("Instala las dependencias de la celda anterior o usa Colab; reinicia el kernel.") from exc
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, f1_score, log_loss, precision_score, recall_score,
)

SEED = 42
keras.utils.set_random_seed(SEED)
WORK_DIR = Path("resultados_semana08_transfer").resolve()
RAW_DIR = WORK_DIR / "data_raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = WORK_DIR / "runs" / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6])
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Keras:", keras.__version__, "| TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("Resultados de esta ejecución:", RUN_DIR)
print("No se ha entrenado ningún modelo todavía.")

# PARTE A — TRABAJO EN CLASE (~60 min)

## Objetivo de esta parte

Antes de terminar la sesión debes haber conseguido:

1. definir un problema de clasificación de 3 clases;
2. descargar y revisar un dataset propio;
3. separar los datos en `train`, `validation` y `test`;
4. cargar EfficientNetB0 preentrenada en ImageNet;
5. congelar el backbone;
6. entrenar una nueva cabeza de clasificación;
7. observar `accuracy` y `val_accuracy`.

> **No evalúes todavía sobre `test`.** Ese conjunto queda reservado para el trabajo en casa.

### Distribución sugerida

- **10 min** — problema y términos de búsqueda;
- **20 min** — descarga, inspección y limpieza;
- **10 min** — separación y carga del dataset;
- **20 min** — transfer learning y primer entrenamiento.

# 1. Define tu problema — EN CLASE

Elige un problema de clasificación de **3 clases**. Las clases deben ser visualmente distinguibles.

Ejemplos: `espresso/cappuccino/latte`, `bus/motorcycle/bicycle`, tres especies de flores, frutas, tipos de calzado o instrumentos musicales.

Evita problemas donde la clase dependa de información que no aparece visualmente en la imagen.

## TODO 1 — Problema y estrategia de búsqueda

Define:

1. las tres clases;
2. dos o tres términos de búsqueda por clase;
3. una breve hipótesis: **¿qué características visuales esperas que use el modelo?**;
4. un posible sesgo o fuente de *data leakage* que podría aparecer al descargar imágenes desde un buscador.

In [ ]:
# Propuesta de problema: vehículos terrestres. Se pueden cambiar ANTES de recolectar.
# Protocolo fijado antes de consultar test: no se cambia según sus resultados.
CANDIDATAS_CAPAS = (20, 40)
EPOCHS_FT = 5
SEARCHES = {
    "bicicleta": ["bicycle side view photograph", "bicycle on street photo", "parked bicycle photo"],
    "motocicleta": ["motorcycle side view photograph", "motorcycle on street photo", "parked motorcycle photo"],
    "autobus": ["city bus side view photograph", "bus on street photo", "parked city bus photo"],
}
DESCARGAR_IMAGENES = False  # True autoriza búsquedas y descargas al ejecutar esa celda.
DATASET_REVISADO = False   # True solo DESPUÉS de inspeccionar el contenido real.
OBSERVACION_LIMPIEZA = ""  # Registra lo observado: no sustituir por una conclusión inventada.
EXCLUIR = {}              # Ejemplo: {"bicicleta/archivo.jpg": "contiene una motocicleta"}
SEARCHES

### Antes de descargar

Escribe debajo de esta celda una respuesta corta:

- ¿Qué características visuales esperas que diferencien tus clases?
- ¿Qué imágenes incorrectas esperas encontrar?
- ¿Qué sesgo o *data leakage* podría aparecer?

### Hipótesis y riesgos definidos antes de entrenar

- **Hipótesis:** las ruedas, el cuadro y los pedales distinguen bicicletas; motor/carenado distinguen motocicletas; carrocería larga y varias ventanas distinguen autobuses.
- **Errores esperables, no observados aún:** juguetes, dibujos, logos, fotos con varios tipos de vehículo, bicicletas eléctricas y objetos muy pequeños.
- **Sesgos posibles:** fondos de estudio frente a calles, marcas de agua, estilos de fotografía y sitios distintos por clase.
- **Fuga de datos:** la misma imagen o variantes casi idénticas podrían aparecer en train y test. Se eliminan duplicados exactos automáticamente y se exige revisar variantes visuales antes de dividir; la deduplicación exacta no garantiza detectar recortes ni otras vistas del mismo vehículo.
- **Datos propios:** aquí significa construir y revisar nuestra colección, como pide la guía. Si usas fotos tuyas, colócalas en las tres carpetas bajo RAW_DIR; no necesitas usar el buscador.
- **Licencias:** no redistribuyas imágenes sin comprobar sus permisos. El registro de fuentes no equivale a una autorización de uso.

# 2. Buscar y descargar imágenes — EN CLASE

La siguiente función busca imágenes y descarga los resultados. Intenta ignorar URLs fallidas, verificar las imágenes, evitar duplicados exactos con un hash y convertirlas a RGB.

> Se solicita contenido con licencia Creative Commons cuando el motor lo soporta. Eso no sustituye la revisión de licencias si el dataset se va a redistribuir.

**Nota de implementación:** el buscador puede devolver enlaces con metadatos de licencia incompletos. El filtro no garantiza permisos; cada fuente queda registrada y debe verificarse antes de redistribuir las fotos.

In [ ]:
EXTENSIONES = {".jpg", ".jpeg", ".png", ".bmp"}

def file_sha256(path):
    with Path(path).open("rb") as archivo:
        return hashlib.file_digest(archivo, "sha256").hexdigest()

def pixel_digest(path):
    with Image.open(path) as imagen:
        rgb = ImageOps.exif_transpose(imagen).convert("RGB")
        return hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()

def download_image_dataset(searches, root, images_per_query=60, target_per_class=150):
    """Busca fotografías, conserva procedencia y no sobrescribe archivos existentes."""
    from ddgs import DDGS  # Solo se necesita cuando se activa la descarga.
    root = Path(root)
    manifest = root.parent / "fuentes.json"
    registros = json.loads(manifest.read_text(encoding="utf-8")) if manifest.exists() else []
    vistos = {file_sha256(p) for p in root.glob("*/*.jpg")}
    session = requests.Session()
    session.headers.update({"User-Agent": "AcademicImageDataset/1.0"})
    fallos = 0
    for clase, queries in searches.items():
        carpeta = root / clase
        carpeta.mkdir(parents=True, exist_ok=True)
        for query in queries:
            if len(list(carpeta.glob("*.jpg"))) >= target_per_class:
                break
            try:
                resultados = DDGS().images(query=query, safesearch="on",
                    max_results=images_per_query, license_image="any")
            except Exception as exc:
                print("Búsqueda no disponible:", clase, type(exc).__name__)
                continue
            for resultado in resultados:
                if len(list(carpeta.glob("*.jpg"))) >= target_per_class:
                    break
                url = resultado.get("image", "")
                if not url.startswith(("https://", "http://")):
                    continue
                try:
                    # Límite de descarga: 12 MB por imagen; no seguir URLs manualmente.
                    with session.get(url, timeout=(5, 15), stream=True) as response:
                        response.raise_for_status()
                        partes, cantidad = [], 0
                        for chunk in response.iter_content(65536):
                            cantidad += len(chunk)
                            if cantidad > 12 * 1024 * 1024:
                                raise ValueError("Imagen demasiado grande")
                            partes.append(chunk)
                    with Image.open(io.BytesIO(b"".join(partes))) as imagen:
                        rgb = ImageOps.exif_transpose(imagen).convert("RGB")
                        if min(rgb.size) < 120:
                            continue
                        # Limitar resolución de almacenamiento, conservando proporciones.
                        rgb.thumbnail((1600, 1600))
                        buffer = io.BytesIO()
                        rgb.save(buffer, format="JPEG", quality=92)
                    datos = buffer.getvalue()
                    digest = hashlib.sha256(datos).hexdigest()
                    if digest in vistos:
                        continue
                    destino = carpeta / (digest + ".jpg")
                    with destino.open("xb") as archivo:
                        archivo.write(datos)
                    vistos.add(digest)
                    registros.append({"archivo": str(destino.relative_to(root)), "clase": clase,
                        "consulta": query, "url_imagen": url, "url_pagina": resultado.get("url", ""),
                        "licencia": "pendiente de revisión individual"})
                except Exception:
                    fallos += 1
            manifest.write_text(json.dumps(registros, indent=2, ensure_ascii=False), encoding="utf-8")
            time.sleep(1)
        print(clase, "archivos:", len(list(carpeta.glob("*.jpg"))))
    session.close()
    print("Descargas fallidas o descartadas:", fallos)
    return registros

def normalizar_foto_local(origen, clase):
    """Crea una copia JPEG orientada en RAW_DIR; conserva el archivo original."""
    if clase not in SEARCHES:
        raise ValueError("Clase no reconocida")
    with Image.open(origen) as imagen:
        rgb = ImageOps.exif_transpose(imagen).convert("RGB")
        rgb.thumbnail((1600, 1600))
        buffer = io.BytesIO()
        rgb.save(buffer, format="JPEG", quality=92)
    datos = buffer.getvalue()
    destino = RAW_DIR / clase / (hashlib.sha256(datos).hexdigest() + ".jpg")
    destino.parent.mkdir(parents=True, exist_ok=True)
    if not destino.exists():
        with destino.open("xb") as archivo:
            archivo.write(datos)
    return destino


## TODO 2 — EN CLASE: construye y limpia tu dataset

Tu objetivo inicial es obtener aproximadamente **100–150 imágenes útiles por clase**.

1. Ejecuta la descarga.
2. Inspecciona muestras aleatorias.
3. Mejora los términos de búsqueda si una clase trae resultados pobres.
4. Elimina imágenes claramente incorrectas o problemáticas.
5. Reporta el número final de imágenes por clase y describe el error más frecuente que encontraste.

> Si la búsqueda trae datos malos, **no aumentes simplemente `max_results`**. Primero mejora los términos de búsqueda.

In [ ]:
for clase in SEARCHES:
    (RAW_DIR / clase).mkdir(exist_ok=True)
if DESCARGAR_IMAGENES:
    fuentes = download_image_dataset(SEARCHES, RAW_DIR)
else:
    print("Descargas desactivadas. Coloca tus imágenes en:", RAW_DIR)
    print("O activa DESCARGAR_IMAGENES y vuelve a ejecutar esta celda.")

In [ ]:
def count_images(root):
    return {clase: sum(p.is_file() and p.suffix.lower() in EXTENSIONES
                      for p in (Path(root) / clase).glob("*"))
            for clase in SEARCHES}
display(pd.DataFrame.from_dict(count_images(RAW_DIR), orient="index", columns=["archivos"]))

if not all(count_images(RAW_DIR).values()):
    raise RuntimeError("Faltan imágenes: recolecta o carga las tres clases antes de continuar.")

## Validación técnica automática de las imágenes

Antes de inspeccionar el contenido del dataset, comprobaremos que **TensorFlow pueda decodificar todas las imágenes**.

Esto resuelve un problema distinto al de la inspección visual:

- **Validación técnica:** ¿el archivo puede ser leído correctamente por TensorFlow?
- **Validación semántica:** ¿la imagen realmente pertenece a la clase indicada?

Una imagen puede abrirse aparentemente bien en otras librerías y aun así fallar durante `model.fit()`. Por eso hacemos esta comprobación usando el mismo decoder que utilizará el pipeline de Keras.

In [ ]:
def validate_images(root):
    """Valida con el decoder real de TensorFlow. Reporta; nunca elimina originales."""
    validos, problemas, duplicados = [], [], []
    hashes = {}
    for clase in sorted(SEARCHES):
        for path in sorted((Path(root) / clase).glob("*")):
            if not path.is_file() or path.suffix.lower() not in EXTENSIONES:
                continue
            relativo = path.relative_to(root).as_posix()
            if relativo in EXCLUIR:
                problemas.append({"archivo": relativo, "motivo": EXCLUIR[relativo]})
                continue
            try:
                with Image.open(path) as pil_image:
                    if pil_image.getexif().get(274, 1) != 1:
                        raise ValueError("EXIF sin normalizar: crea una copia con normalizar_foto_local(path, clase)")
                imagen = tf.io.decode_image(tf.io.read_file(str(path)), channels=3, expand_animations=False)
                forma = imagen.numpy().shape
                if len(forma) != 3 or min(forma[:2]) < 120:
                    raise ValueError("Resolución insuficiente")
                digest = pixel_digest(path)
                if digest in hashes:
                    previo = hashes[digest]
                    if previo["clase"] != clase:
                        raise ValueError("Misma imagen etiquetada en clases distintas; revisar y excluir una copia")
                    duplicados.append({"archivo": relativo, "copia_de": previo["archivo"]})
                    continue
                registro = {"archivo": relativo, "clase": clase,
                            "sha256": file_sha256(path), "pixel_sha256": digest}
                hashes[digest] = registro
                validos.append(registro)
            except Exception as exc:
                problemas.append({"archivo": relativo, "motivo": str(exc).splitlines()[0]})
    return validos, problemas, duplicados

inventario, problemas, duplicados = validate_images(RAW_DIR)
display(pd.DataFrame(problemas))
display(pd.DataFrame(duplicados))
print("No se borró ningún original. Se utilizarán únicamente imágenes válidas y no excluidas.")

### Verificación

Ejecuta nuevamente la función. El resultado esperado es:

```text
Imágenes inválidas eliminadas: 0
```

Si todavía aparece alguna imagen inválida, vuelve a ejecutar hasta obtener cero antes de continuar.

### Verificación técnica y limpieza sin borrar originales

Los archivos ilegibles y los duplicados se excluyen del inventario de entrenamiento, pero permanecen en la carpeta original. Revisa el reporte; corrige las etiquetas ambiguas mediante EXCLUIR. Vuelve a ejecutar la validación después de cambiar las exclusiones. Si detectas versiones recortadas o casi idénticas, exclúyelas antes del split.

La validación automática no puede decidir si una foto está bien etiquetada.

Las fotos locales con orientación EXIF distinta de 1 se excluyen: TensorFlow no aplica esa rotación automáticamente. Usa `normalizar_foto_local(ruta, clase)` para crear una copia orientada en la carpeta correcta y repite la validación. El original permanece intacto. Las imágenes descargadas ya se guardan orientadas.

**Cambio técnico documentado:** la guía original borra archivos ilegibles. Aquí se excluyen del inventario y se registran, preservando tus originales. Por eso la comprobación posterior valida el inventario útil, no espera eliminar archivos del disco.

In [ ]:
assert len({r["pixel_sha256"] for r in inventario}) == len(inventario)
pd.DataFrame(inventario).to_csv(RUN_DIR / "inventario.csv", index=False)
pd.DataFrame(problemas, columns=["archivo", "motivo"]).to_csv(RUN_DIR / "exclusiones.csv", index=False)
pd.DataFrame(duplicados, columns=["archivo", "copia_de"]).to_csv(RUN_DIR / "duplicados.csv", index=False)
conteos = {c: sum(r["clase"] == c for r in inventario) for c in SEARCHES}
display(pd.DataFrame.from_dict(conteos, orient="index", columns=["válidas únicas"]))
if any(n < 100 for n in conteos.values()):
    warnings.warn("La guía propone 100–150 imágenes útiles por clase; aún no alcanzas ese objetivo.")
if any(n < 20 for n in conteos.values()):
    raise ValueError("Se requieren al menos 20 imágenes válidas por clase para este experimento.")

# 3. Inspección y limpieza semántica — EN CLASE

La etapa anterior confirmó que los archivos son **técnicamente válidos**.

Ahora viene una tarea diferente: comprobar si los datos son **semánticamente correctos**.

Busca especialmente:

- imágenes que no correspondan a la clase;
- dibujos, logos o diagramas si tu objetivo son fotografías;
- imágenes con texto que pueda revelar artificialmente la etiqueta;
- duplicados o imágenes casi idénticas;
- fondos que estén correlacionados con una clase;
- diferencias sistemáticas de estilo o calidad entre clases.

> Que TensorFlow pueda abrir una imagen no significa que sea un buen dato de entrenamiento.

In [ ]:
def show_random_images(root, records, n_per_class=6):
    rng = random.Random(SEED)
    fig, axes = plt.subplots(3, n_per_class, figsize=(3*n_per_class, 9), squeeze=False)
    for row, clase in enumerate(sorted(SEARCHES)):
        archivos = [r["archivo"] for r in records if r["clase"] == clase]
        elegidos = rng.sample(archivos, min(n_per_class, len(archivos)))
        for col, ax in enumerate(axes[row]):
            if col < len(elegidos):
                with Image.open(Path(root) / elegidos[col]) as imagen:
                    ax.imshow(ImageOps.exif_transpose(imagen).convert("RGB"))
                ax.set_title(clase + "\n" + Path(elegidos[col]).name[:12])
            ax.axis("off")
    fig.tight_layout()
    fig.savefig(RUN_DIR / "muestra_dataset.png", dpi=140)
    plt.show()

show_random_images(RAW_DIR, inventario)

### Checkpoint del dataset

Antes de continuar, asegúrate de que:

- cada carpeta contiene imágenes realmente asociadas a su clase;
- no predominan logos, dibujos o texto que revele la etiqueta;
- no hay una diferencia de fondo/estilo demasiado obvia entre clases;
- el número de imágenes por clase es razonablemente balanceado.

Registra aquí el número de imágenes eliminadas y el problema de calidad más frecuente.

### Revisión humana necesaria

Examina también otras imágenes, no únicamente la muestra. En la celda de configuración:
1. Registra en EXCLUIR las imágenes incorrectas o casi duplicadas.
2. Completa OBSERVACION_LIMPIEZA con lo que efectivamente encontraste.
3. Marca DATASET_REVISADO=True cuando termines y repite la validación.

No se declara que el dataset esté limpio sin inspeccionarlo. El siguiente control impide entrenar inadvertidamente sobre una colección sin revisar.

# 4. Separar train, validation y test — EN CLASE

> **Concepto clave — tres conjuntos, tres funciones**
>
> - **Train:** se utiliza para actualizar los pesos del modelo.
> - **Validation:** se utiliza durante el desarrollo para decidir épocas, `learning_rate`, número de capas a descongelar, etc.
> - **Test:** se reserva para medir el rendimiento final. No debe guiar las decisiones de entrenamiento.

Usaremos **70 % entrenamiento, 15 % validación y 15 % prueba**.

In [ ]:
def partition_records(records, seed=42, train_ratio=0.70, val_ratio=0.15):
    """Split determinista por clase y sin modificar ni borrar la fuente."""
    if train_ratio <= 0 or val_ratio <= 0 or train_ratio + val_ratio >= 1:
        raise ValueError("Proporciones no válidas")
    rng = random.Random(seed)
    salida = []
    for clase in sorted({r["clase"] for r in records}):
        grupo = sorted((dict(r) for r in records if r["clase"] == clase), key=lambda r: r["archivo"])
        rng.shuffle(grupo)
        n = len(grupo)
        a, b = int(n * train_ratio), int(n * (train_ratio + val_ratio))
        if min(a, b-a, n-b) < 1:
            raise ValueError("Muy pocas imágenes por clase")
        for nombre, elementos in (("train", grupo[:a]), ("validation", grupo[a:b]), ("test", grupo[b:])):
            salida.extend({**r, "split": nombre} for r in elementos)
    hashes = [r["pixel_sha256"] for r in salida]
    if len(hashes) != len(set(hashes)):
        raise ValueError("Duplicados entre particiones")
    return salida

def split_dataset(source_root, records, workspace, seed=42):
    filas = partition_records(records, seed)
    firma = hashlib.sha256(json.dumps(filas, sort_keys=True).encode()).hexdigest()[:16]
    destino = Path(workspace) / ("dataset_" + firma)
    destino.mkdir(parents=True, exist_ok=True)
    for fila in filas:
        origen = Path(source_root) / fila["archivo"]
        if file_sha256(origen) != fila["sha256"]:
            raise ValueError("Una imagen cambió después de validar; vuelve a crear el inventario.")
        nuevo = destino / fila["split"] / fila["clase"] / Path(fila["archivo"]).name
        nuevo.parent.mkdir(parents=True, exist_ok=True)
        if nuevo.exists():
            if file_sha256(nuevo) != fila["sha256"]:
                raise ValueError("El split existente fue modificado: " + str(nuevo))
        else:
            shutil.copy2(origen, nuevo)
    pd.DataFrame(filas).to_csv(destino / "split_manifest.csv", index=False)
    return destino, filas

if not DATASET_REVISADO or not OBSERVACION_LIMPIEZA.strip():
    raise RuntimeError("Inspecciona el dataset; registra OBSERVACION_LIMPIEZA y marca DATASET_REVISADO=True.")
SPLIT_DIR, split_records = split_dataset(RAW_DIR, inventario, WORK_DIR, SEED)
display(pd.crosstab(pd.Series([r["clase"] for r in split_records], name="clase"),
                    pd.Series([r["split"] for r in split_records], name="partición")))
(RUN_DIR / "limpieza.json").write_text(json.dumps({
    "observacion": OBSERVACION_LIMPIEZA, "conteos": conteos,
    "exclusiones": EXCLUIR, "split": str(SPLIT_DIR)}, ensure_ascii=False, indent=2), encoding="utf-8")

### Comprobación conceptual

Explica brevemente: ¿por qué usar repetidamente el conjunto de prueba para escoger épocas, `learning_rate` o capas a descongelar termina contaminando nuestra estimación de desempeño?

### Respuesta
Si elegimos épocas, learning rate o capas después de observar test, sus etiquetas influyen indirectamente en nuestras decisiones. Su métrica deja de ser una evaluación independiente y puede resultar optimista.

Train actualiza pesos; validation selecciona épocas y capas. En este notebook se conserva el orden de la guía: hay una evaluación inicial en test y otra tras el fine-tuning. Para no usar la primera como señal de ajuste, los candidatos (20/40 capas), tasas y máximo de épocas se fijan antes, y el ganador se elige exclusivamente por validation. Si cambiamos el protocolo tras ver los errores de test, necesitaremos otro conjunto independiente para estimar el nuevo desempeño.

Las proporciones son aproximadamente 70/15/15 por clase debido al redondeo entero. El manifiesto permite comprobar qué archivo quedó en cada partición.


# 5. Construir los datasets de Keras — EN CLASE

`image_dataset_from_directory()` construye un `tf.data.Dataset` usando el nombre de las carpetas como clases.

Observa que:

- `train` se mezcla (`shuffle=True`);
- `validation` y `test` mantienen un orden estable;
- todas las imágenes se redimensionan a `224×224` para EfficientNetB0.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
class_names = sorted(SEARCHES)
NUM_CLASSES = len(class_names)

def load_dataset(split_name, shuffle):
    return keras.utils.image_dataset_from_directory(
        SPLIT_DIR / split_name, class_names=class_names,
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="int",
        shuffle=shuffle, seed=SEED if shuffle else None,
    )

train_ds = load_dataset("train", True).prefetch(tf.data.AUTOTUNE)
val_ds = load_dataset("validation", False).prefetch(tf.data.AUTOTUNE)
print("Orden explícito de clases:", class_names)
# Test conserva el mismo orden; se evalúa por primera vez en la sección 7.
test_raw = load_dataset("test", False)
test_paths = list(test_raw.file_paths)
test_ds = test_raw.prefetch(tf.data.AUTOTUNE)
# EfficientNetB0 de Keras incluye Rescaling: entregar float [0,255], NO dividir por 255.
batch_images, batch_labels = next(iter(train_ds))
assert tuple(batch_images.shape[1:]) == (224, 224, 3)
assert float(tf.reduce_min(batch_images)) >= 0 and float(tf.reduce_max(batch_images)) <= 255

# 6. Primer modelo: transfer learning — EN CLASE

Usaremos **EfficientNetB0** preentrenada en ImageNet. Primero congelaremos completamente la red base y entrenaremos solo una nueva cabeza de clasificación.

> **Concepto clave — Data augmentation**
>
> `RandomFlip`, `RandomRotation` y `RandomZoom` generan variaciones plausibles durante el entrenamiento. No crean información nueva, pero ayudan a que el modelo dependa menos de detalles accidentales de las imágenes descargadas.

> **Concepto clave — GlobalAveragePooling2D**
>
> En lugar de aplanar todos los mapas de características con `Flatten`, `GlobalAveragePooling2D` resume cada canal en un solo valor. Así reducimos mucho el número de parámetros de la cabeza de clasificación.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal", seed=SEED),
    layers.RandomRotation(0.08, seed=SEED+1),
    layers.RandomZoom(0.10, seed=SEED+2),
], name="augmentation")

base_model = keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet", input_shape=(*IMG_SIZE, 3))
base_model.trainable = False
BASE_NAME = base_model.name
inputs = keras.Input(shape=(*IMG_SIZE, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)  # Conserva estadísticas BatchNorm en ambas etapas.
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2, seed=SEED)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = keras.Model(inputs, outputs, name="vehiculos_transfer")
assert not base_model.trainable_weights
model.summary()

## TODO 3 — EN CLASE: entrena la cabeza de clasificación

La base EfficientNet está congelada. Tu trabajo es configurar y entrenar la nueva cabeza.

> **Concepto clave — EarlyStopping**
>
> `EarlyStopping` detiene el entrenamiento cuando `val_loss` deja de mejorar. Con `restore_best_weights=True`, Keras recupera los pesos de la mejor época observada en validación.

Antes de ejecutar, escribe una predicción corta:

- ¿esperas que `train_accuracy` suba rápidamente?
- ¿qué señal en las curvas indicaría overfitting?

Completa únicamente los espacios `TODO` del código siguiente.

### Predicción antes de entrenar
Espero que train_accuracy mejore relativamente rápido porque los filtros de ImageNet ya representan bordes, texturas y formas. Esto es una hipótesis, no un resultado observado.

Una señal de sobreajuste sería que la pérdida de entrenamiento siga disminuyendo mientras val_loss aumenta, acompañada de una brecha creciente entre train_accuracy y val_accuracy. EarlyStopping selecciona pesos por val_loss, no por test. Completo Adam con 1e-3, accuracy, monitor val_loss y val_ds.


In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])
history = model.fit(train_ds, validation_data=val_ds, epochs=10,
    callbacks=[keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=2, restore_best_weights=True)])
BASELINE_PATH = RUN_DIR / "modelo_congelado.keras"
model.save(BASELINE_PATH)
pd.DataFrame(history.history).to_csv(RUN_DIR / "historial_congelado.csv", index_label="epoca")

### Curvas de aprendizaje — EN CLASE si el tiempo lo permite

Grafica `accuracy`, `val_accuracy`, `loss` y `val_loss`.

Si el tiempo de clase termina antes, **puedes completar estas curvas en casa**.  
Lo obligatorio antes de salir es haber entrenado el primer modelo y tener sus métricas de entrenamiento/validación.

In [ ]:
def plot_history(history, title, filename):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    epochs = np.arange(1, len(history.history["loss"]) + 1)
    for ax, metric in zip(axes, ["accuracy", "loss"]):
        ax.plot(epochs, history.history[metric], label="train")
        ax.plot(epochs, history.history["val_" + metric], label="validation")
        ax.set(xlabel="Época", ylabel=metric, title=title + " · " + metric)
        ax.legend()
        ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(RUN_DIR / filename, dpi=150)
    plt.show()

plot_history(history, "Base congelada", "curvas_congelado.png")

# PUNTO DE CONTROL — FIN DEL TRABAJO EN CLASE

Antes de continuar, verifica que tu equipo tenga:

- [ ] 3 clases definidas;
- [ ] términos de búsqueda documentados;
- [ ] imágenes descargadas;
- [ ] dataset inspeccionado y parcialmente limpiado;
- [ ] separación `train / validation / test`;
- [ ] EfficientNetB0 con pesos de ImageNet;
- [ ] backbone congelado;
- [ ] nueva cabeza de clasificación;
- [ ] al menos un entrenamiento completado;
- [ ] `accuracy` y `val_accuracy` reportadas.

## Hasta aquí llega el trabajo de clase.

**No uses todavía el conjunto de test para tomar decisiones.**

---

# PARTE B — TRABAJO EN CASA

A partir de este punto debes completar el análisis de manera autónoma.

Ahora sí vas a:

1. evaluar el modelo en `test`;
2. estudiar la matriz de confusión y métricas por clase;
3. realizar fine-tuning;
4. comparar el modelo congelado contra el modelo ajustado;
5. analizar errores;
6. escribir las conclusiones.

# 7. Evaluación del primer modelo — EN CASA

Ahora sí utilizaremos el conjunto de prueba.

> **Concepto clave — No todo es accuracy**
>
> - **Precision:** de las imágenes que el modelo predijo como una clase, ¿cuántas eran correctas?
> - **Recall:** de las imágenes que realmente pertenecían a una clase, ¿cuántas encontró?
> - **F1:** combina precision y recall.
> - **Matriz de confusión:** muestra qué clases se están confundiendo entre sí.

Guarda esta evaluación: será nuestra línea base antes del fine-tuning.

In [ ]:
# Se conserva la evaluación inicial en test donde la pide la guía.
# Las candidatas 20/40, el LR y las épocas se fijaron antes de ver esta métrica.
val_before = model.evaluate(val_ds, verbose=0, return_dict=True)
test_loss_before, test_acc_before = model.evaluate(test_ds, verbose=0)
baseline = model
print(f"Test accuracy antes del fine-tuning: {test_acc_before:.4f}")
print("No modificar el protocolo ni elegir capas utilizando este valor.")


In [ ]:
def get_predictions(model, dataset):
    y_true = np.concatenate([labels.numpy() for _, labels in dataset])
    probabilities = model.predict(dataset, verbose=0)
    if probabilities.shape != (len(y_true), NUM_CLASSES):
        raise ValueError("Dimensiones inesperadas")
    np.testing.assert_allclose(probabilities.sum(axis=1), 1, atol=1e-5)
    return y_true, probabilities.argmax(axis=1), probabilities

def evaluate_classifier(model, dataset, phase):
    real, pred, probabilities = get_predictions(model, dataset)
    metrics = {"fase": phase, "accuracy": float(accuracy_score(real, pred)),
        "precision_macro": float(precision_score(real, pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(real, pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(real, pred, average="macro", zero_division=0)),
        "log_loss": float(log_loss(real, probabilities, labels=range(NUM_CLASSES)))}
    report = classification_report(real, pred, labels=range(NUM_CLASSES),
        target_names=class_names, output_dict=True, zero_division=0)
    cm = confusion_matrix(real, pred, labels=range(NUM_CLASSES))
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title("Test — " + phase)
    fig.tight_layout()
    fig.savefig(RUN_DIR / ("confusion_" + phase + ".png"), dpi=150)
    plt.show()
    display(pd.DataFrame(report).T)
    (RUN_DIR / ("reporte_" + phase + ".json")).write_text(
        json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
    # Interpretación numérica real; la explicación visual se reserva a la revisión.
    f1s = np.array([report[c]["f1-score"] for c in class_names])
    mejores = [c for c, f in zip(class_names, f1s) if np.isclose(f, f1s.max())]
    fuera = cm.copy()
    np.fill_diagonal(fuera, 0)
    pares = [(class_names[i], class_names[j], int(fuera[i,j]))
             for i,j in zip(*np.where(fuera == fuera.max())) if fuera[i,j] > 0]
    print("Mejor F1 por clase (incluye empates):", mejores)
    print("Confusiones más frecuentes real → predicha:", pares or "ninguna")
    print("Diferencia entre máximo y mínimo recall:", max(report[c]["recall"] for c in class_names)
          - min(report[c]["recall"] for c in class_names))
    return metrics, real, pred, probabilities
# Resultado y matriz del primer modelo, en su sección original.
before, y_true, y_pred, probabilities = evaluate_classifier(model, test_ds, "congelado")


## TODO 4 — EN CASA: interpreta la evaluación

Utilizando `classification_report` y la matriz de confusión, responde:

1. ¿Cuál clase obtuvo mejores resultados?
2. ¿Cuál fue la principal confusión?
3. ¿La accuracy global oculta algún problema por clase?
4. ¿Qué cambiarías primero en el **dataset** antes de cambiar el modelo?

### Respuesta e interpretación reproducible
1. La celda anterior imprime la clase o clases con **mejor F1**, incluyendo empates. No se escribe un nombre sin haber evaluado.
2. Imprime los pares **real → predicha** con más errores fuera de la diagonal.
3. La diferencia entre recalls y el informe por clase permiten ver si la accuracy global oculta una clase con peor cobertura; también debe considerarse el número de ejemplos de cada clase.
4. Revisaría primero las etiquetas y la representatividad de las clases confundidas, duplicados y correlaciones con fondos. Son hipótesis de diagnóstico, no causas demostradas sin observar fotos. Un nuevo experimento motivado por errores de test necesitaría otra evaluación independiente.

Los nombres y valores concretos se generan al ejecutar el informe; todavía no hay resultados empíricos.


# 8. Fine-tuning — EN CASA

Ahora permitiremos que una pequeña parte de EfficientNet se adapte al nuevo problema.

> **Concepto clave — Fine-tuning**
>
> En la fase anterior, los pesos del backbone no cambiaron. Ahora descongelaremos únicamente una parte final del modelo y la actualizaremos con un **learning rate mucho menor**.

> **Concepto clave — Batch Normalization**
>
> EfficientNet contiene capas `BatchNormalization`, que mantienen estadísticas internas aprendidas durante el entrenamiento. En datasets pequeños conviene mantenerlas congeladas durante fine-tuning para evitar actualizaciones inestables de esas estadísticas.

Después de cambiar qué capas son entrenables, es necesario **recompilar** el modelo.

In [ ]:
# Se usa el protocolo fijado en TODO 1, sin modificarlo a partir de test.
assert 3 <= EPOCHS_FT <= 8

def configure_fine_tuning(candidate, n_unfrozen):
    backbone = candidate.get_layer(BASE_NAME)
    if not 1 <= n_unfrozen <= len(backbone.layers):
        raise ValueError("Número de capas inválido")
    backbone.trainable = True
    for i, layer in enumerate(backbone.layers):
        layer.trainable = i >= len(backbone.layers) - n_unfrozen
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    assert all(not layer.trainable for layer in backbone.layers
               if isinstance(layer, layers.BatchNormalization))
    candidate.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        loss=keras.losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])
    return sum(layer.trainable and bool(layer.weights) for layer in backbone.layers)

print("Se comparan las candidatas predefinidas", CANDIDATAS_CAPAS, "solo mediante val_loss.")

## TODO 5 — EN CASA: realiza fine-tuning

Recompila y continúa el entrenamiento.

Usa como punto de partida:

- `Adam(learning_rate=1e-5)`;
- `SparseCategoricalCrossentropy`;
- `accuracy`;
- entre **3 y 8 épocas** adicionales;
- `EarlyStopping` sobre `val_loss`.

Tu decisión principal aquí es **cuántas capas descongelar**. Justifícala usando el desempeño de validación, no el conjunto de prueba.

## Fine-tuning y learning rate

Cada candidato empieza desde el mismo archivo del modelo congelado. Se evalúan 20 y 40 capas finales (BatchNorm permanece congelada), con Adam LR=1e-5 y hasta cinco épocas adicionales. EarlyStopping puede terminar antes.

El protocolo se fijó antes de la evaluación inicial de test. La menor val_loss elige las capas; un empate favorece la alternativa más pequeña. La tasa baja limita alteraciones bruscas de características preentrenadas. No es una garantía de mejora.

**Qué significa el número de capas:** incluye capas sin pesos; por eso también se informa cuántas capas con pesos quedan realmente entrenables.

In [ ]:
ft_results = []
histories_ft = {}
for n_unfrozen in CANDIDATAS_CAPAS:
    keras.utils.set_random_seed(SEED)
    candidate = keras.models.load_model(BASELINE_PATH)
    n_weight_layers = configure_fine_tuning(candidate, n_unfrozen)
    h = candidate.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT,
        callbacks=[keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=2, restore_best_weights=True)])
    validation = candidate.evaluate(val_ds, verbose=0, return_dict=True)
    candidate_path = RUN_DIR / f"fine_tuning_{n_unfrozen}.keras"
    candidate.save(candidate_path)
    histories_ft[n_unfrozen] = h
    ft_results.append({"capas_finales": n_unfrozen, "capas_con_pesos_entrenables": n_weight_layers,
        "val_loss": float(validation["loss"]), "val_accuracy": float(validation["accuracy"]),
        "epocas": len(h.history["loss"]), "modelo": str(candidate_path)})
    pd.DataFrame(h.history).to_csv(RUN_DIR / f"historial_ft_{n_unfrozen}.csv", index_label="epoca")
    del candidate

best_ft = min(ft_results, key=lambda row: (row["val_loss"], row["capas_finales"]))
N_UNFROZEN = best_ft["capas_finales"]
model_ft = keras.models.load_model(best_ft["modelo"])
history_ft = histories_ft[N_UNFROZEN]
display(pd.DataFrame(ft_results))
pd.DataFrame(ft_results).to_csv(RUN_DIR / "seleccion_por_validation.csv", index=False)
print("Capas elegidas por validation:", N_UNFROZEN)
plot_history(history_ft, f"Fine-tuning ({N_UNFROZEN} capas finales)", "curvas_fine_tuning.png")
# Elegir entre congelado y fine-tuning también se hace con validation.
modelo_recomendado = "fine_tuning" if best_ft["val_loss"] < val_before["loss"] else "congelado"
print("Modelo recomendado exclusivamente por val_loss:", modelo_recomendado)

# 9. Comparación final — EN CASA

Compara el desempeño del mismo sistema **antes y después** del fine-tuning.

Una mejora pequeña también puede ser válida. Si el resultado empeora, no significa que el experimento “salió mal”: debes interpretar por qué pudo ocurrir.

In [ ]:
# Comparación final: se reutiliza el resultado congelado de la sección 7.
# Ninguna decisión de entrenamiento depende del test.
after, y_true_ft, y_pred_ft, probabilities_ft = evaluate_classifier(model_ft, test_ds, "fine_tuning")
test_loss_after, test_acc_after = model_ft.evaluate(test_ds, verbose=0)
np.testing.assert_array_equal(y_true, y_true_ft)
comparison = pd.DataFrame([before, after])
display(comparison)
comparison.to_csv(RUN_DIR / "comparacion_test.csv", index=False)
print(f"Antes del fine-tuning: {test_acc_before:.4f}")
print(f"Después del fine-tuning: {test_acc_after:.4f}")
print(f"Diferencia: {test_acc_after-test_acc_before:+.4f}")
print("No cambiar el protocolo usando esta comparación de test.")


In [ ]:
prediction_rows = []
for phase, truth, predicted, probs in (
    ("congelado", y_true, y_pred, probabilities),
    ("fine_tuning", y_true_ft, y_pred_ft, probabilities_ft),
):
    for path, real, pred, p in zip(test_paths, truth, predicted, probs):
        prediction_rows.append({"fase": phase, "archivo": str(Path(path).relative_to(SPLIT_DIR)),
            "real": class_names[int(real)], "predicha": class_names[int(pred)],
            "confianza": float(p[pred]), **{f"p_{c}": float(v) for c,v in zip(class_names,p)}})
predictions_df = pd.DataFrame(prediction_rows)
predictions_df.to_csv(RUN_DIR / "predicciones_test.csv", index=False)
(RUN_DIR / "configuracion.json").write_text(json.dumps({
    "seed": SEED, "clases": class_names, "input_size": IMG_SIZE, "split": str(SPLIT_DIR),
    "learning_rate_cabeza": 1e-3, "learning_rate_ft": 1e-5,
    "capas_elegidas": N_UNFROZEN, "recomendado_por_validation": modelo_recomendado,
    "tensorflow": tf.__version__, "keras": keras.__version__,
}, ensure_ascii=False, indent=2), encoding="utf-8")

# 10. Analiza los errores — EN CASA

Una métrica no explica por qué falla un modelo. Debes inspeccionar ejemplos clasificados incorrectamente.

In [ ]:
def show_mistakes(paths, truth, predicted, probs, max_images=12):
    indices = np.flatnonzero(truth != predicted)
    # Primero los errores de mayor confianza: diagnóstico, no estimación imparcial.
    indices = sorted(indices, key=lambda i: float(probs[i, predicted[i]]), reverse=True)[:max_images]
    if not indices:
        print("No hay errores en este test. No fabricar ejemplos para completar una tabla.")
        return []
    fig, axes = plt.subplots(int(np.ceil(len(indices) / 4)), 4,
                            figsize=(16, 4 * int(np.ceil(len(indices) / 4))), squeeze=False)
    rows = []
    for ax in axes.ravel():
        ax.axis("off")
    for ax, i in zip(axes.ravel(), indices):
        with Image.open(paths[i]) as image:
            ax.imshow(ImageOps.exif_transpose(image).convert("RGB"))
        ax.set_title(f"#{i}: real {class_names[truth[i]]}\npred {class_names[predicted[i]]} ({probs[i,predicted[i]]:.2f})")
        rows.append({"id_test": int(i), "archivo": str(Path(paths[i]).relative_to(SPLIT_DIR)),
            "clase_real": class_names[truth[i]], "prediccion": class_names[predicted[i]],
            "confianza": float(probs[i,predicted[i]]),
            "causa_probable": "REQUIERE INSPECCIÓN VISUAL",
            "mejora_propuesta": "REQUIERE ANÁLISIS"})
    fig.tight_layout()
    fig.savefig(RUN_DIR / "errores_fine_tuning.png", dpi=150)
    plt.show()
    return rows

error_rows = show_mistakes(test_paths, y_true_ft, y_pred_ft, probabilities_ft)
pd.DataFrame(error_rows, columns=["id_test","archivo","clase_real","prediccion","confianza",
    "causa_probable","mejora_propuesta"]).to_csv(RUN_DIR / "analisis_errores.csv", index=False)
if len(error_rows) < 5:
    print("Hay menos de cinco errores: reporta todos y documenta esta limitación.")
    print("No alteres etiquetas ni empeores el modelo para fabricar cinco fallos.")

## TODO 6 — EN CASA: análisis de errores y conclusiones

Selecciona al menos **5 errores** del modelo y analiza su causa probable.

| Imagen/error | Clase real | Predicción | Causa probable | ¿Cómo lo mejorarías? |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |
| 5 | | | | |

Posibles causas: imagen ambigua, etiqueta incorrecta, objeto pequeño, oclusión, fondo engañoso, clase visualmente similar, mala calidad, sesgo del dataset o error razonable del modelo.

Después, escribe una conclusión breve respondiendo:

1. ¿Qué aprendiste al construir el dataset?
2. ¿Cuál fue el principal problema de calidad?
3. ¿Cuánto cambió el desempeño después del fine-tuning?
4. ¿El fine-tuning ayudó? ¿Por qué crees que sí o no?
5. ¿Qué mejorarías primero: datos, entrenamiento o arquitectura?

## Análisis de errores y conclusiones empíricas

El código muestra hasta 12 errores reales y crea analisis_errores.csv. **Después de mirar las imágenes**, completa causa_probable y mejora_propuesta para al menos cinco errores si existen. Una confianza alta no demuestra por sí sola la causa del fallo.

Este análisis no puede resolverse honestamente antes de disponer del dataset y sus predicciones. Si hay menos de cinco fallos, incluye todos los reales, informa la limitación y consulta al docente cómo complementar el requisito, sin alterar el test ni inventar errores.

La celda siguiente produce una conclusión cuantitativa basada en resultados reales, conserva la observación de limpieza y advierte qué parte de la entrega aún requiere revisión.

La tabla original se conserva. El CSV y la siguiente visualización contienen las filas reales para trasladarlas a esa tabla después de inspeccionarlas.

In [ ]:
# Tabla de errores reales: causas y mejoras solo después de inspección visual.
display(pd.DataFrame(error_rows))
# Resumen empírico generado a partir de esta ejecución, no valores preescritos.
delta = 100 * (after["accuracy"] - before["accuracy"])
conclusion = (
    f"Dataset: {conteos}. Observación de limpieza: {OBSERVACION_LIMPIEZA}\n\n"
    f"Fine-tuning seleccionado con {N_UNFROZEN} capas finales según validation. "
    f"Accuracy test: {before['accuracy']:.4f} → {after['accuracy']:.4f}; "
    f"cambio: {delta:+.2f} puntos. F1 macro: {before['f1_macro']:.4f} → {after['f1_macro']:.4f}.\n\n"
    f"Modelo recomendado por validation: {modelo_recomendado}. "
    "Una mejora de test en una sola partición no prueba superioridad general. "
    "Primero revisaría calidad y representatividad de los datos. "
    "Las causas individuales de errores se deben justificar visualmente en analisis_errores.csv."
)
print(conclusion)
(RUN_DIR / "conclusiones.md").write_text(conclusion, encoding="utf-8")



# Extensión opcional — Prueba con una imagen externa

Busca una imagen que **no esté en tu dataset** y prueba el modelo. Esta parte no cuenta como un TODO principal, pero sirve como comprobación cualitativa de generalización.

In [ ]:
# Opcional: inferencia con una imagen externa local, sin llamadas a servidores.
EXTERNAL_IMAGE_PATH = None
if EXTERNAL_IMAGE_PATH:
    with Image.open(EXTERNAL_IMAGE_PATH) as image:
        rgb = ImageOps.exif_transpose(image).convert("RGB")
        x = np.asarray(rgb.resize(IMG_SIZE), dtype="float32")[None, ...]
        chosen = model_ft if modelo_recomendado == "fine_tuning" else baseline
        p = chosen.predict(x, verbose=0)[0]
        plt.imshow(rgb)
        plt.axis("off")
        plt.title(f"{class_names[p.argmax()]} — {p.max():.2f}")
        plt.show()

# 🏠 Cierre conceptual — ENTREGA

Como parte del **TODO 6**, explica con tus propias palabras la diferencia entre:

- **feature extraction / transfer learning con base congelada**;
- **fine-tuning**;
- **entrenar una CNN desde cero**.

No describas solo el código. Explica **qué parámetros se están aprendiendo y de dónde viene el conocimiento inicial**.

# Cierre conceptual — respuestas

- **Extracción de características:** el backbone conserva los filtros aprendidos en ImageNet; solamente se aprenden los pesos de la nueva cabeza. El conocimiento inicial viene de millones de patrones visuales del entrenamiento previo.
- **Fine-tuning:** se ajusta también una parte de los filtros preentrenados con un learning rate bajo. En este ejercicio BatchNorm conserva sus estadísticas; la cabeza sigue aprendiendo.
- **CNN desde cero:** todos los pesos comienzan sin el conocimiento de ImageNet. Normalmente exige más datos y cómputo. No se ha entrenado una red desde cero en este taller.
- **Qué se aprende al construir datos:** la calidad de las etiquetas, la variedad y la separación de fuentes importan tanto como la arquitectura. La experiencia concreta de limpieza está registrada por el estudiante, no inferida a partir de una accuracy.
- **Learning rate:** 1e-3 adapta la cabeza nueva; 1e-5 limita cambios en el conocimiento previo. Estos valores son puntos de partida de la guía, no óptimos universales.

Referencias primarias: [guía de transfer learning de Keras](https://keras.io/guides/transfer_learning/) y [EfficientNet](https://keras.io/api/applications/efficientnet/).

# Reto opcional

Elige **uno**:

**A. Más datos:** duplica el número de imágenes manteniendo el modelo igual.  
**B. Otro backbone:** prueba MobileNetV3, ResNet50 o EfficientNetV2B0.  
**C. Dataset difícil:** agrega una cuarta clase visualmente parecida.  
**D. Ablation:** entrena sin `data_augmentation` y compara.

Cambia **una sola variable a la vez**.

### Reto elegido: D — Ablation
Se propone quitar únicamente data_augmentation, manteniendo backbone congelado, cabeza, split, optimizador y máximo de épocas. El código opcional compara validation con el primer entrenamiento; **no vuelve a utilizar test** para elegir el sistema. Es exploratorio y no sustituye el experimento principal. La ejecución se deja desactivada para evitar entrenamiento adicional accidental.


In [ ]:
EJECUTAR_RETO_ABLATION = False
if EJECUTAR_RETO_ABLATION:
    keras.utils.set_random_seed(SEED)
    base_ablation = keras.applications.EfficientNetB0(
        include_top=False, weights="imagenet", input_shape=(*IMG_SIZE, 3))
    base_ablation.trainable = False
    inputs_ablation = keras.Input(shape=(*IMG_SIZE, 3))
    features = base_ablation(inputs_ablation, training=False)
    features = layers.GlobalAveragePooling2D()(features)
    features = layers.Dropout(0.2, seed=SEED)(features)
    outputs_ablation = layers.Dense(NUM_CLASSES, activation="softmax")(features)
    ablation = keras.Model(inputs_ablation, outputs_ablation)
    ablation.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss=keras.losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])
    # Dataset nuevo para reiniciar también su secuencia de mezcla.
    ablation_train = load_dataset("train", True).prefetch(tf.data.AUTOTUNE)
    history_ablation = ablation.fit(ablation_train, validation_data=val_ds, epochs=10,
        callbacks=[keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=2, restore_best_weights=True)])
    val_ablation = ablation.evaluate(val_ds, verbose=0, return_dict=True)
    tabla_ablation = pd.DataFrame([
        {"experimento": "con_augmentation", **val_before},
        {"experimento": "sin_augmentation", **val_ablation},
    ])
    display(tabla_ablation)
    tabla_ablation.to_csv(RUN_DIR / "ablation_validation.csv", index=False)
    plot_history(history_ablation, "Sin augmentation", "curvas_ablation.png")
    print("Un solo entrenamiento no estima la variabilidad entre semillas.")
else:
    print("Reto opcional implementado, no ejecutado.")


# ✅ Checklist de entrega

Antes de entregar verifica:

### Evidencia del trabajo en clase
- [ ] Problema y clases definidos.
- [ ] Dataset construido y revisado.
- [ ] Train / validation / test creados.
- [ ] Primer modelo con backbone congelado entrenado.

### Trabajo en casa
- [ ] Evaluación sobre test.
- [ ] Matriz de confusión.
- [ ] Precision / recall / F1 interpretados.
- [ ] Fine-tuning ejecutado.
- [ ] Comparación antes vs. después del fine-tuning.
- [ ] Al menos 5 errores analizados.
- [ ] Conclusión final escrita.

### Opcional
- [ ] Imagen externa.
- [ ] Experimento adicional / reto.

# Rúbrica sugerida

| Componente | Peso |
|---|---:|
| TODO 1 — Problema y estrategia de búsqueda | 10% |
| TODO 2 — Construcción, inspección y limpieza del dataset | 20% |
| Split y pipeline de datos | 10% |
| TODO 3 — Transfer learning + curvas | 20% |
| TODO 4 — Evaluación e interpretación | 10% |
| TODO 5 — Fine-tuning y comparación | 15% |
| TODO 6 — Análisis de errores + conclusión conceptual | 15% |
| **Total** | **100%** |

## Cambios y estado de ejecución

- Conservados el nombre, las 52 celdas originales en orden, todos sus enunciados Markdown e identificadores.
- Completados los parámetros del entrenamiento, las curvas, el fine-tuning y las respuestas conceptuales. Se añaden interpretación numérica, exportaciones y ablation opcional.
- Instalación y recolección explícitas; controles de orientación EXIF, duplicados e imágenes inválidas. La limpieza y el split no borran tus datos.
- Se conserva la evaluación inicial en test en la sección 7. Las opciones de fine-tuning se fijan antes y solo se seleccionan con validation; no se usa test para cambiar el protocolo.
- EfficientNet recibe píxeles [0,255], BatchNorm permanece congelada y cada candidato parte del mismo modelo guardado.
- Las salidas históricas se han limpiado para no atribuir a esta ejecución resultados ajenos.
- **Pendiente externo:** imágenes reales, inspección, entrenamiento, cinco errores (si existen), causas justificadas visualmente y guardar las salidas. No se afirma haber cumplido los puntos empíricos del checklist.
